# M4 · Métricas de transcripción 


## 1. Clonar repositorio

In [ ]:
RAMA = "LuisM4"

!git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo

In [ ]:
%cd /content/repo
!git pull
%cd M4/metricas_transcripcion
!pip install -r requirements.txt -q

## 2. Montar Drive y definir `PROJECT_ROOT`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
with open("/content/repo/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")

manifest = PROJECT_ROOT / "M4/data/imagenes/manifest.json"
print("Manifiesto:", manifest, "| existe:", manifest.exists())
print("Caches de transcripcion disponibles:")
for p in sorted((PROJECT_ROOT / "M4/vision/predictions").glob("transcripciones_*.json")):
    print("  ", p.relative_to(PROJECT_ROOT))

## 3. Probar el módulo con datos simulados (opcional)

Genera transcripciones con errores conocidos y comprueba que las métricas los miden. **No son resultados del sistema**: van a `M4/outputs/metricas_transcripcion/simulado/`.

In [ ]:
!python run_metricas.py --simular

## 4. Medir las transcripciones reales

Un cache de transcripciones por nivel (y, si existe, su cache NER en el mismo orden). Las rutas son relativas a `PROJECT_ROOT`. También se pueden fijar en `entradas` de `config.yaml`.

In [ ]:
TRANSCRIPCIONES = [
    "M4/vision/predictions/transcripciones_vlm_limpio.json",
    "M4/vision/predictions/transcripciones_vlm_foto_realista.json",
]
candidatos_ner = [
    "M2/predictions/vision_vlm_limpio.json",
    "M2/predictions/vision_vlm_foto_realista.json",
]
NER = []
if all((PROJECT_ROOT / n).exists() for n in candidatos_ner):
    NER = candidatos_ner
    print("Predicciones NER encontradas en M2/predictions: se incluirán en la auditoría.")
else:
    print("Predicciones NER no encontradas aún: evaluando solo transcripción (WER/CER).")

args = " ".join(f'"{t}"' for t in TRANSCRIPCIONES)
args_ner = (" --ner " + " ".join(f'"{n}"' for n in NER)) if NER else ""
if TRANSCRIPCIONES:
    !python run_metricas.py --transcripciones {args}{args_ner}
else:
    print("Agrega al menos un cache en TRANSCRIPCIONES")


## 5. Ver el resumen

In [ ]:
from IPython.display import Markdown, display

for sub in ("", "simulado"):
    ruta = PROJECT_ROOT / "M4/outputs/metricas_transcripcion" / sub / "resumen.md"
    if ruta.exists():
        display(Markdown(ruta.read_text(encoding="utf-8")))